# KSB1 — Partidas individuais de centro de custo (controlling) — Análise Exploratória

**Tabela:** `dev_procurement.corp_curated.vw_ds_ctrl_ksb1`
**Transação SAP:** KSB1 · **Objeto:** **VIEW**
**Colunas:** 40 · **Clustering declarado:** _(nenhum — é view)_

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória entre elas.

## Alertas de partida _(extraídos dos comentários do `DESCRIBE`)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | `tp_objeto_controlling` é declarado como derivado dos 2 primeiros caracteres de `cod_objeto_controlling`, **mas essa coluna não existe na tabela**. Origem do dado a esclarecer. |
| 🔴 ALTA | **Não existe coluna de centro/planta.** Só `cod_centro_custo` (centro de custo), que é conceito diferente. Filtro por centro não é aplicável diretamente. |
| 🟡 MEDIA | **Colunas redundantes declaradas:** `desc_conta_razao` resolve para o mesmo texto de `desc_classe_custo`; `desc_contrapartida` resolve para o mesmo de `desc_conta_contrapartida`. |
| 🟡 MEDIA | `cod_objeto_relatorio` é preenchido com o **centro de custo**, não com o objeto de controlling — divergente da semântica do SAP. |
| 🟡 MEDIA | **3 colunas de data são `string`** — normalizar antes de comparar. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por `cod_centro_custo` — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas e timestamps |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| **18** | **Análises específicas da transação** |
| 19 | Amostra ampliada |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados

Este objeto é uma **view**. `DESCRIBE DETAIL` e `DESCRIBE HISTORY` só funcionam em tabelas
Delta — por isso ficaram comentados, para não interromper o **Run All**.

Se precisar do histórico, rode-os na tabela física de origem.

In [0]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_ds_ctrl_ksb1;

In [0]:
-- 1.1 DETALHE E HISTORICO — nao se aplicam a view.
-- Descomente somente se o objeto for convertido em tabela Delta.
-- DESCRIBE DETAIL dev_procurement.corp_curated.vw_ds_ctrl_ksb1;
-- DESCRIBE HISTORY dev_procurement.corp_curated.vw_ds_ctrl_ksb1 LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                        AS linhas,
       COUNT(DISTINCT CONCAT_WS('|', `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `cod_item_partida_individual_ledger`)) AS chaves_completas,
       COUNT(DISTINCT `cod_centro_custo`)                 AS valores_de_cod_centro_custo,
       COUNT(DISTINCT `num_lancamento_contabil`)                 AS num_lancamento_contabil_distintos
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1;

## 3. Distribuição por `cod_centro_custo`

⚠️ **Esta tabela não possui coluna de centro/planta.**

O campo mais próximo é o centro de custo, que é **conceito diferente** de centro/planta.

Confirme com o time antes de usar esta coluna como recorte de teste.

In [0]:
-- 3. DISTRIBUICAO POR cod_centro_custo
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro_custo` AS STRING)), ''), '(vazio)') AS `cod_centro_custo`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro_custo` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional multiplicando linhas.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1),
g AS (
  SELECT 'num_lancamento_contabil' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
UNION ALL
  SELECT 'num_lancamento_contabil + cod_item_partida_individual_ledger' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_lancamento_contabil`, `cod_item_partida_individual_ledger` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil + cod_item_partida_individual_ledger' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `cod_item_partida_individual_ledger` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_lancamento_contabil + cod_item_partida_individual_ledger`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_lancamento_contabil`, `cod_item_partida_individual_ledger`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY `num_lancamento_contabil`, `cod_item_partida_individual_ledger`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (
  SELECT `num_lancamento_contabil`, `cod_item_partida_individual_ledger` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `num_lancamento_contabil`, `cod_item_partida_individual_ledger` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 c JOIN dup ON c.`num_lancamento_contabil` <=> dup.`num_lancamento_contabil` AND c.`cod_item_partida_individual_ledger` <=> dup.`cod_item_partida_individual_ledger`
),
agg AS (
  SELECT `num_lancamento_contabil`, `cod_item_partida_individual_ledger`,
         COUNT(DISTINCT `cod_conta_razao`) AS `cod_conta_razao`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `vl_moeda_global`) AS `vl_moeda_global`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `qt_lancada`) AS `qt_lancada`,
         COUNT(DISTINCT `dt_lancamento`) AS `dt_lancamento`,
         COUNT(DISTINCT `num_referencia`) AS `num_referencia`,
         COUNT(DISTINCT `desc_texto_item`) AS `desc_texto_item`,
         COUNT(DISTINCT `desc_classe_custo`) AS `desc_classe_custo`,
         COUNT(DISTINCT `tp_operacao_co`) AS `tp_operacao_co`,
         COUNT(DISTINCT `cod_debito_credito`) AS `cod_debito_credito`,
         COUNT(DISTINCT `cod_conta_contrapartida`) AS `cod_conta_contrapartida`,
         COUNT(DISTINCT `dt_documento`) AS `dt_documento`,
         COUNT(DISTINCT `desc_conta_razao`) AS `desc_conta_razao`,
         COUNT(DISTINCT `desc_conta_contrapartida`) AS `desc_conta_contrapartida`,
         COUNT(DISTINCT `desc_contrapartida`) AS `desc_contrapartida`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `num_exercicio_fiscal`) AS `num_exercicio_fiscal`,
         COUNT(DISTINCT `cod_ledger_fonte`) AS `cod_ledger_fonte`,
         COUNT(DISTINCT `cod_moeda_transacao`) AS `cod_moeda_transacao`,
         COUNT(DISTINCT `cod_moeda_empresa`) AS `cod_moeda_empresa`,
         COUNT(DISTINCT `cod_moeda_objeto_co`) AS `cod_moeda_objeto_co`,
         COUNT(DISTINCT `cod_moeda_relatorio`) AS `cod_moeda_relatorio`,
         COUNT(DISTINCT `nm_usuario_criou_lancamento_contabil`) AS `nm_usuario_criou_lancamento_contabil`,
         COUNT(DISTINCT `cod_objeto_relatorio`) AS `cod_objeto_relatorio`,
         COUNT(DISTINCT `cod_periodo_contabil`) AS `cod_periodo_contabil`,
         COUNT(DISTINCT `tp_conta`) AS `tp_conta`,
         COUNT(DISTINCT `tp_lancamento_contabil`) AS `tp_lancamento_contabil`,
         COUNT(DISTINCT `tp_objeto_controlling`) AS `tp_objeto_controlling`,
         COUNT(DISTINCT `vl_moeda_transacao`) AS `vl_moeda_transacao`,
         COUNT(DISTINCT `vl_moeda_empresa`) AS `vl_moeda_empresa`,
         COUNT(DISTINCT `vl_moeda_objeto`) AS `vl_moeda_objeto`,
         COUNT(DISTINCT `tp_transacao_comercial`) AS `tp_transacao_comercial`,
         COUNT(DISTINCT `cod_area_contabilidade_custos`) AS `cod_area_contabilidade_custos`,
         COUNT(DISTINCT `cod_unidade_medida_basica`) AS `cod_unidade_medida_basica`,
         COUNT(DISTINCT `dt_ingestao`) AS `dt_ingestao`
  FROM d GROUP BY `num_lancamento_contabil`, `cod_item_partida_individual_ledger`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(38,
    'cod_conta_razao', MAX(`cod_conta_razao`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'dt_criacao', MAX(`dt_criacao`),
    'vl_moeda_global', MAX(`vl_moeda_global`),
    'cod_material', MAX(`cod_material`),
    'desc_material', MAX(`desc_material`),
    'qt_lancada', MAX(`qt_lancada`),
    'dt_lancamento', MAX(`dt_lancamento`),
    'num_referencia', MAX(`num_referencia`),
    'desc_texto_item', MAX(`desc_texto_item`),
    'desc_classe_custo', MAX(`desc_classe_custo`),
    'tp_operacao_co', MAX(`tp_operacao_co`),
    'cod_debito_credito', MAX(`cod_debito_credito`),
    'cod_conta_contrapartida', MAX(`cod_conta_contrapartida`),
    'dt_documento', MAX(`dt_documento`),
    'desc_conta_razao', MAX(`desc_conta_razao`),
    'desc_conta_contrapartida', MAX(`desc_conta_contrapartida`),
    'desc_contrapartida', MAX(`desc_contrapartida`),
    'cod_empresa', MAX(`cod_empresa`),
    'num_exercicio_fiscal', MAX(`num_exercicio_fiscal`),
    'cod_ledger_fonte', MAX(`cod_ledger_fonte`),
    'cod_moeda_transacao', MAX(`cod_moeda_transacao`),
    'cod_moeda_empresa', MAX(`cod_moeda_empresa`),
    'cod_moeda_objeto_co', MAX(`cod_moeda_objeto_co`),
    'cod_moeda_relatorio', MAX(`cod_moeda_relatorio`),
    'nm_usuario_criou_lancamento_contabil', MAX(`nm_usuario_criou_lancamento_contabil`),
    'cod_objeto_relatorio', MAX(`cod_objeto_relatorio`),
    'cod_periodo_contabil', MAX(`cod_periodo_contabil`),
    'tp_conta', MAX(`tp_conta`),
    'tp_lancamento_contabil', MAX(`tp_lancamento_contabil`),
    'tp_objeto_controlling', MAX(`tp_objeto_controlling`),
    'vl_moeda_transacao', MAX(`vl_moeda_transacao`),
    'vl_moeda_empresa', MAX(`vl_moeda_empresa`),
    'vl_moeda_objeto', MAX(`vl_moeda_objeto`),
    'tp_transacao_comercial', MAX(`tp_transacao_comercial`),
    'cod_area_contabilidade_custos', MAX(`cod_area_contabilidade_custos`),
    'cod_unidade_medida_basica', MAX(`cod_unidade_medida_basica`),
    'dt_ingestao', MAX(`dt_ingestao`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Em validação anterior, esta análise revelou 7 colunas 100% nulas no Datalake —
uma delas preenchida em 97,9% dos registros do SAP. Este erro **não aparece** em amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, a coluna 'zeros_ou_false' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1),
perf AS (
  SELECT stack(40,
    'cod_item_partida_individual_ledger', 'string', COUNT_IF(`cod_item_partida_individual_ledger` IS NULL), COUNT_IF(`cod_item_partida_individual_ledger` IS NOT NULL AND lower(trim(`cod_item_partida_individual_ledger`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_item_partida_individual_ledger`) RLIKE '^0+([.,]0+)?$'),
    'cod_conta_razao', 'string', COUNT_IF(`cod_conta_razao` IS NULL), COUNT_IF(`cod_conta_razao` IS NOT NULL AND lower(trim(`cod_conta_razao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_conta_razao`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'string', COUNT_IF(`dt_criacao` IS NULL), COUNT_IF(`dt_criacao` IS NOT NULL AND lower(trim(`dt_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_criacao`) RLIKE '^0+([.,]0+)?$'),
    'vl_moeda_global', 'double', COUNT_IF(`vl_moeda_global` IS NULL), 0L, COUNT_IF(`vl_moeda_global` = 0),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'qt_lancada', 'decimal(23,3)', COUNT_IF(`qt_lancada` IS NULL), 0L, COUNT_IF(`qt_lancada` = 0),
    'dt_lancamento', 'string', COUNT_IF(`dt_lancamento` IS NULL), COUNT_IF(`dt_lancamento` IS NOT NULL AND lower(trim(`dt_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_lancamento`) RLIKE '^0+([.,]0+)?$'),
    'num_referencia', 'string', COUNT_IF(`num_referencia` IS NULL), COUNT_IF(`num_referencia` IS NOT NULL AND lower(trim(`num_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_referencia`) RLIKE '^0+([.,]0+)?$'),
    'desc_texto_item', 'string', COUNT_IF(`desc_texto_item` IS NULL), COUNT_IF(`desc_texto_item` IS NOT NULL AND lower(trim(`desc_texto_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_texto_item`) RLIKE '^0+([.,]0+)?$'),
    'desc_classe_custo', 'string', COUNT_IF(`desc_classe_custo` IS NULL), COUNT_IF(`desc_classe_custo` IS NOT NULL AND lower(trim(`desc_classe_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_classe_custo`) RLIKE '^0+([.,]0+)?$'),
    'tp_operacao_co', 'string', COUNT_IF(`tp_operacao_co` IS NULL), COUNT_IF(`tp_operacao_co` IS NOT NULL AND lower(trim(`tp_operacao_co`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_operacao_co`) RLIKE '^0+([.,]0+)?$'),
    'cod_debito_credito', 'string', COUNT_IF(`cod_debito_credito` IS NULL), COUNT_IF(`cod_debito_credito` IS NOT NULL AND lower(trim(`cod_debito_credito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_debito_credito`) RLIKE '^0+([.,]0+)?$'),
    'cod_conta_contrapartida', 'string', COUNT_IF(`cod_conta_contrapartida` IS NULL), COUNT_IF(`cod_conta_contrapartida` IS NOT NULL AND lower(trim(`cod_conta_contrapartida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_conta_contrapartida`) RLIKE '^0+([.,]0+)?$'),
    'dt_documento', 'string', COUNT_IF(`dt_documento` IS NULL), COUNT_IF(`dt_documento` IS NOT NULL AND lower(trim(`dt_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_documento`) RLIKE '^0+([.,]0+)?$'),
    'desc_conta_razao', 'string', COUNT_IF(`desc_conta_razao` IS NULL), COUNT_IF(`desc_conta_razao` IS NOT NULL AND lower(trim(`desc_conta_razao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_conta_razao`) RLIKE '^0+([.,]0+)?$'),
    'desc_conta_contrapartida', 'string', COUNT_IF(`desc_conta_contrapartida` IS NULL), COUNT_IF(`desc_conta_contrapartida` IS NOT NULL AND lower(trim(`desc_conta_contrapartida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_conta_contrapartida`) RLIKE '^0+([.,]0+)?$'),
    'desc_contrapartida', 'string', COUNT_IF(`desc_contrapartida` IS NULL), COUNT_IF(`desc_contrapartida` IS NOT NULL AND lower(trim(`desc_contrapartida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_contrapartida`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'num_exercicio_fiscal', 'string', COUNT_IF(`num_exercicio_fiscal` IS NULL), COUNT_IF(`num_exercicio_fiscal` IS NOT NULL AND lower(trim(`num_exercicio_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_exercicio_fiscal`) RLIKE '^0+([.,]0+)?$'),
    'cod_ledger_fonte', 'string', COUNT_IF(`cod_ledger_fonte` IS NULL), COUNT_IF(`cod_ledger_fonte` IS NOT NULL AND lower(trim(`cod_ledger_fonte`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ledger_fonte`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda_transacao', 'string', COUNT_IF(`cod_moeda_transacao` IS NULL), COUNT_IF(`cod_moeda_transacao` IS NOT NULL AND lower(trim(`cod_moeda_transacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_transacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda_empresa', 'string', COUNT_IF(`cod_moeda_empresa` IS NULL), COUNT_IF(`cod_moeda_empresa` IS NOT NULL AND lower(trim(`cod_moeda_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda_objeto_co', 'string', COUNT_IF(`cod_moeda_objeto_co` IS NULL), COUNT_IF(`cod_moeda_objeto_co` IS NOT NULL AND lower(trim(`cod_moeda_objeto_co`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_objeto_co`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda_relatorio', 'string', COUNT_IF(`cod_moeda_relatorio` IS NULL), COUNT_IF(`cod_moeda_relatorio` IS NOT NULL AND lower(trim(`cod_moeda_relatorio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_relatorio`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario_criou_lancamento_contabil', 'string', COUNT_IF(`nm_usuario_criou_lancamento_contabil` IS NULL), COUNT_IF(`nm_usuario_criou_lancamento_contabil` IS NOT NULL AND lower(trim(`nm_usuario_criou_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario_criou_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'num_lancamento_contabil', 'string', COUNT_IF(`num_lancamento_contabil` IS NULL), COUNT_IF(`num_lancamento_contabil` IS NOT NULL AND lower(trim(`num_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'cod_objeto_relatorio', 'string', COUNT_IF(`cod_objeto_relatorio` IS NULL), COUNT_IF(`cod_objeto_relatorio` IS NOT NULL AND lower(trim(`cod_objeto_relatorio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_objeto_relatorio`) RLIKE '^0+([.,]0+)?$'),
    'cod_periodo_contabil', 'string', COUNT_IF(`cod_periodo_contabil` IS NULL), COUNT_IF(`cod_periodo_contabil` IS NOT NULL AND lower(trim(`cod_periodo_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_periodo_contabil`) RLIKE '^0+([.,]0+)?$'),
    'tp_conta', 'string', COUNT_IF(`tp_conta` IS NULL), COUNT_IF(`tp_conta` IS NOT NULL AND lower(trim(`tp_conta`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_conta`) RLIKE '^0+([.,]0+)?$'),
    'tp_lancamento_contabil', 'string', COUNT_IF(`tp_lancamento_contabil` IS NULL), COUNT_IF(`tp_lancamento_contabil` IS NOT NULL AND lower(trim(`tp_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'tp_objeto_controlling', 'string', COUNT_IF(`tp_objeto_controlling` IS NULL), COUNT_IF(`tp_objeto_controlling` IS NOT NULL AND lower(trim(`tp_objeto_controlling`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_objeto_controlling`) RLIKE '^0+([.,]0+)?$'),
    'vl_moeda_transacao', 'double', COUNT_IF(`vl_moeda_transacao` IS NULL), 0L, COUNT_IF(`vl_moeda_transacao` = 0),
    'vl_moeda_empresa', 'double', COUNT_IF(`vl_moeda_empresa` IS NULL), 0L, COUNT_IF(`vl_moeda_empresa` = 0),
    'vl_moeda_objeto', 'double', COUNT_IF(`vl_moeda_objeto` IS NULL), 0L, COUNT_IF(`vl_moeda_objeto` = 0),
    'tp_transacao_comercial', 'string', COUNT_IF(`tp_transacao_comercial` IS NULL), COUNT_IF(`tp_transacao_comercial` IS NOT NULL AND lower(trim(`tp_transacao_comercial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_transacao_comercial`) RLIKE '^0+([.,]0+)?$'),
    'cod_area_contabilidade_custos', 'string', COUNT_IF(`cod_area_contabilidade_custos` IS NULL), COUNT_IF(`cod_area_contabilidade_custos` IS NOT NULL AND lower(trim(`cod_area_contabilidade_custos`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_area_contabilidade_custos`) RLIKE '^0+([.,]0+)?$'),
    'cod_unidade_medida_basica', 'string', COUNT_IF(`cod_unidade_medida_basica` IS NULL), COUNT_IF(`cod_unidade_medida_basica` IS NOT NULL AND lower(trim(`cod_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'dt_ingestao', 'date', COUNT_IF(`dt_ingestao` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1),
card AS (
  SELECT stack(40,
    'cod_item_partida_individual_ledger', 'string', approx_count_distinct(`cod_item_partida_individual_ledger`),
    'cod_conta_razao', 'string', approx_count_distinct(`cod_conta_razao`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'dt_criacao', 'string', approx_count_distinct(`dt_criacao`),
    'vl_moeda_global', 'double', approx_count_distinct(`vl_moeda_global`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'qt_lancada', 'decimal(23,3)', approx_count_distinct(`qt_lancada`),
    'dt_lancamento', 'string', approx_count_distinct(`dt_lancamento`),
    'num_referencia', 'string', approx_count_distinct(`num_referencia`),
    'desc_texto_item', 'string', approx_count_distinct(`desc_texto_item`),
    'desc_classe_custo', 'string', approx_count_distinct(`desc_classe_custo`),
    'tp_operacao_co', 'string', approx_count_distinct(`tp_operacao_co`),
    'cod_debito_credito', 'string', approx_count_distinct(`cod_debito_credito`),
    'cod_conta_contrapartida', 'string', approx_count_distinct(`cod_conta_contrapartida`),
    'dt_documento', 'string', approx_count_distinct(`dt_documento`),
    'desc_conta_razao', 'string', approx_count_distinct(`desc_conta_razao`),
    'desc_conta_contrapartida', 'string', approx_count_distinct(`desc_conta_contrapartida`),
    'desc_contrapartida', 'string', approx_count_distinct(`desc_contrapartida`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'num_exercicio_fiscal', 'string', approx_count_distinct(`num_exercicio_fiscal`),
    'cod_ledger_fonte', 'string', approx_count_distinct(`cod_ledger_fonte`),
    'cod_moeda_transacao', 'string', approx_count_distinct(`cod_moeda_transacao`),
    'cod_moeda_empresa', 'string', approx_count_distinct(`cod_moeda_empresa`),
    'cod_moeda_objeto_co', 'string', approx_count_distinct(`cod_moeda_objeto_co`),
    'cod_moeda_relatorio', 'string', approx_count_distinct(`cod_moeda_relatorio`),
    'nm_usuario_criou_lancamento_contabil', 'string', approx_count_distinct(`nm_usuario_criou_lancamento_contabil`),
    'num_lancamento_contabil', 'string', approx_count_distinct(`num_lancamento_contabil`),
    'cod_objeto_relatorio', 'string', approx_count_distinct(`cod_objeto_relatorio`),
    'cod_periodo_contabil', 'string', approx_count_distinct(`cod_periodo_contabil`),
    'tp_conta', 'string', approx_count_distinct(`tp_conta`),
    'tp_lancamento_contabil', 'string', approx_count_distinct(`tp_lancamento_contabil`),
    'tp_objeto_controlling', 'string', approx_count_distinct(`tp_objeto_controlling`),
    'vl_moeda_transacao', 'double', approx_count_distinct(`vl_moeda_transacao`),
    'vl_moeda_empresa', 'double', approx_count_distinct(`vl_moeda_empresa`),
    'vl_moeda_objeto', 'double', approx_count_distinct(`vl_moeda_objeto`),
    'tp_transacao_comercial', 'string', approx_count_distinct(`tp_transacao_comercial`),
    'cod_area_contabilidade_custos', 'string', approx_count_distinct(`cod_area_contabilidade_custos`),
    'cod_unidade_medida_basica', 'string', approx_count_distinct(`cod_unidade_medida_basica`),
    'dt_ingestao', 'date', approx_count_distinct(`dt_ingestao`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'num_exercicio_fiscal' AS coluna, CAST(`num_exercicio_fiscal` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `num_exercicio_fiscal` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_periodo_contabil' AS coluna, CAST(`cod_periodo_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_periodo_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_conta' AS coluna, CAST(`tp_conta` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `tp_conta` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_lancamento_contabil' AS coluna, CAST(`tp_lancamento_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `tp_lancamento_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_debito_credito' AS coluna, CAST(`cod_debito_credito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_debito_credito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_operacao_co' AS coluna, CAST(`tp_operacao_co` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `tp_operacao_co` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_objeto_controlling' AS coluna, CAST(`tp_objeto_controlling` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `tp_objeto_controlling` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_transacao_comercial' AS coluna, CAST(`tp_transacao_comercial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `tp_transacao_comercial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_area_contabilidade_custos' AS coluna, CAST(`cod_area_contabilidade_custos` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_area_contabilidade_custos` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda_transacao' AS coluna, CAST(`cod_moeda_transacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_moeda_transacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda_empresa' AS coluna, CAST(`cod_moeda_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_moeda_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_ledger_fonte' AS coluna, CAST(`cod_ledger_fonte` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_ledger_fonte` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_unidade_medida_basica' AS coluna, CAST(`cod_unidade_medida_basica` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1 GROUP BY `cod_unidade_medida_basica` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil dos campos numéricos

A tabela tem **4 colunas `double`** entre as numéricas analisadas — exigem tolerância de 0,005 na comparação com o SAP.

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(5,
    'vl_moeda_global', 'double', COUNT(`vl_moeda_global`), CAST(MIN(`vl_moeda_global`) AS DOUBLE), CAST(MAX(`vl_moeda_global`) AS DOUBLE), CAST(AVG(`vl_moeda_global`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_global`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_global` < 0), COUNT_IF(`vl_moeda_global` = 0),
    'qt_lancada', 'decimal(23,3)', COUNT(`qt_lancada`), CAST(MIN(`qt_lancada`) AS DOUBLE), CAST(MAX(`qt_lancada`) AS DOUBLE), CAST(AVG(`qt_lancada`) AS DOUBLE), CAST(percentile_approx(`qt_lancada`, 0.5) AS DOUBLE), COUNT_IF(`qt_lancada` < 0), COUNT_IF(`qt_lancada` = 0),
    'vl_moeda_transacao', 'double', COUNT(`vl_moeda_transacao`), CAST(MIN(`vl_moeda_transacao`) AS DOUBLE), CAST(MAX(`vl_moeda_transacao`) AS DOUBLE), CAST(AVG(`vl_moeda_transacao`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_transacao`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_transacao` < 0), COUNT_IF(`vl_moeda_transacao` = 0),
    'vl_moeda_empresa', 'double', COUNT(`vl_moeda_empresa`), CAST(MIN(`vl_moeda_empresa`) AS DOUBLE), CAST(MAX(`vl_moeda_empresa`) AS DOUBLE), CAST(AVG(`vl_moeda_empresa`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_empresa`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_empresa` < 0), COUNT_IF(`vl_moeda_empresa` = 0),
    'vl_moeda_objeto', 'double', COUNT(`vl_moeda_objeto`), CAST(MIN(`vl_moeda_objeto`) AS DOUBLE), CAST(MAX(`vl_moeda_objeto`) AS DOUBLE), CAST(AVG(`vl_moeda_objeto`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_objeto`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_objeto` < 0), COUNT_IF(`vl_moeda_objeto` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(5,
    'vl_moeda_global', CAST(SUM(`vl_moeda_global`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_global`), 2) AS STRING), COUNT(`vl_moeda_global`),
    'qt_lancada', CAST(SUM(`qt_lancada`) AS DOUBLE), CAST(ROUND(SUM(`qt_lancada`), 2) AS STRING), COUNT(`qt_lancada`),
    'vl_moeda_transacao', CAST(SUM(`vl_moeda_transacao`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_transacao`), 2) AS STRING), COUNT(`vl_moeda_transacao`),
    'vl_moeda_empresa', CAST(SUM(`vl_moeda_empresa`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_empresa`), 2) AS STRING), COUNT(`vl_moeda_empresa`),
    'vl_moeda_objeto', CAST(SUM(`vl_moeda_objeto`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_objeto`), 2) AS STRING), COUNT(`vl_moeda_objeto`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
)
ORDER BY coluna;

## 12. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
Mesma data, formato diferente — já gerou 16.773 falsos positivos em validação anterior.

In [0]:
-- DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(3,
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL OR trim(`dt_criacao`) = ''), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_criacao`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END), MAX(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END),
    'dt_lancamento', COUNT_IF(`dt_lancamento` IS NULL OR trim(`dt_lancamento`) = ''), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_lancamento`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_lancamento`) NOT IN ('', '00000000') THEN `dt_lancamento` END), MAX(CASE WHEN trim(`dt_lancamento`) NOT IN ('', '00000000') THEN `dt_lancamento` END),
    'dt_documento', COUNT_IF(`dt_documento` IS NULL OR trim(`dt_documento`) = ''), COUNT_IF(trim(`dt_documento`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_documento`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_documento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_documento`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_documento`) NOT IN ('', '00000000') THEN `dt_documento` END), MAX(CASE WHEN trim(`dt_documento`) NOT IN ('', '00000000') THEN `dt_documento` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
)
ORDER BY coluna;

## 12.1 Datas em tipo nativo

In [0]:
-- DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(1,
    'dt_ingestao', COUNT_IF(`dt_ingestao` IS NULL), CAST(MIN(`dt_ingestao`) AS STRING), CAST(MAX(`dt_ingestao`) AS STRING), COUNT(DISTINCT `dt_ingestao`), COUNT_IF(`dt_ingestao` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(8,
    'num_lancamento_contabil', 'string', COUNT_IF(CAST(`num_lancamento_contabil` AS STRING) IS NULL OR trim(CAST(`num_lancamento_contabil` AS STRING)) = ''), MIN(length(trim(CAST(`num_lancamento_contabil` AS STRING)))), MAX(length(trim(CAST(`num_lancamento_contabil` AS STRING)))), COUNT_IF(trim(CAST(`num_lancamento_contabil` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_lancamento_contabil` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_lancamento_contabil` AS STRING)), '^0+', '')),
    'cod_item_partida_individual_ledger', 'string', COUNT_IF(CAST(`cod_item_partida_individual_ledger` AS STRING) IS NULL OR trim(CAST(`cod_item_partida_individual_ledger` AS STRING)) = ''), MIN(length(trim(CAST(`cod_item_partida_individual_ledger` AS STRING)))), MAX(length(trim(CAST(`cod_item_partida_individual_ledger` AS STRING)))), COUNT_IF(trim(CAST(`cod_item_partida_individual_ledger` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_item_partida_individual_ledger` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_item_partida_individual_ledger` AS STRING)), '^0+', '')),
    'cod_empresa', 'string', COUNT_IF(CAST(`cod_empresa` AS STRING) IS NULL OR trim(CAST(`cod_empresa` AS STRING)) = ''), MIN(length(trim(CAST(`cod_empresa` AS STRING)))), MAX(length(trim(CAST(`cod_empresa` AS STRING)))), COUNT_IF(trim(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_empresa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', '')),
    'cod_conta_razao', 'string', COUNT_IF(CAST(`cod_conta_razao` AS STRING) IS NULL OR trim(CAST(`cod_conta_razao` AS STRING)) = ''), MIN(length(trim(CAST(`cod_conta_razao` AS STRING)))), MAX(length(trim(CAST(`cod_conta_razao` AS STRING)))), COUNT_IF(trim(CAST(`cod_conta_razao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_conta_razao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_conta_razao` AS STRING)), '^0+', '')),
    'cod_conta_contrapartida', 'string', COUNT_IF(CAST(`cod_conta_contrapartida` AS STRING) IS NULL OR trim(CAST(`cod_conta_contrapartida` AS STRING)) = ''), MIN(length(trim(CAST(`cod_conta_contrapartida` AS STRING)))), MAX(length(trim(CAST(`cod_conta_contrapartida` AS STRING)))), COUNT_IF(trim(CAST(`cod_conta_contrapartida` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_conta_contrapartida` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_conta_contrapartida` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_objeto_relatorio', 'string', COUNT_IF(CAST(`cod_objeto_relatorio` AS STRING) IS NULL OR trim(CAST(`cod_objeto_relatorio` AS STRING)) = ''), MIN(length(trim(CAST(`cod_objeto_relatorio` AS STRING)))), MAX(length(trim(CAST(`cod_objeto_relatorio` AS STRING)))), COUNT_IF(trim(CAST(`cod_objeto_relatorio` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_objeto_relatorio` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_objeto_relatorio` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real,
não granularidade adicional.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_item_partida_individual_ledger` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`vl_moeda_global` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`qt_lancada` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`desc_texto_item` AS STRING), ''), COALESCE(CAST(`desc_classe_custo` AS STRING), ''), COALESCE(CAST(`tp_operacao_co` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`cod_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`desc_conta_razao` AS STRING), ''), COALESCE(CAST(`desc_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`desc_contrapartida` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`cod_ledger_fonte` AS STRING), ''), COALESCE(CAST(`cod_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_moeda_empresa` AS STRING), ''), COALESCE(CAST(`cod_moeda_objeto_co` AS STRING), ''), COALESCE(CAST(`cod_moeda_relatorio` AS STRING), ''), COALESCE(CAST(`nm_usuario_criou_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_objeto_relatorio` AS STRING), ''), COALESCE(CAST(`cod_periodo_contabil` AS STRING), ''), COALESCE(CAST(`tp_conta` AS STRING), ''), COALESCE(CAST(`tp_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`tp_objeto_controlling` AS STRING), ''), COALESCE(CAST(`vl_moeda_transacao` AS STRING), ''), COALESCE(CAST(`vl_moeda_empresa` AS STRING), ''), COALESCE(CAST(`vl_moeda_objeto` AS STRING), ''), COALESCE(CAST(`tp_transacao_comercial` AS STRING), ''), COALESCE(CAST(`cod_area_contabilidade_custos` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_item_partida_individual_ledger` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`vl_moeda_global` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`qt_lancada` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`desc_texto_item` AS STRING), ''), COALESCE(CAST(`desc_classe_custo` AS STRING), ''), COALESCE(CAST(`tp_operacao_co` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`cod_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`desc_conta_razao` AS STRING), ''), COALESCE(CAST(`desc_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`desc_contrapartida` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`cod_ledger_fonte` AS STRING), ''), COALESCE(CAST(`cod_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_moeda_empresa` AS STRING), ''), COALESCE(CAST(`cod_moeda_objeto_co` AS STRING), ''), COALESCE(CAST(`cod_moeda_relatorio` AS STRING), ''), COALESCE(CAST(`nm_usuario_criou_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_objeto_relatorio` AS STRING), ''), COALESCE(CAST(`cod_periodo_contabil` AS STRING), ''), COALESCE(CAST(`tp_conta` AS STRING), ''), COALESCE(CAST(`tp_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`tp_objeto_controlling` AS STRING), ''), COALESCE(CAST(`vl_moeda_transacao` AS STRING), ''), COALESCE(CAST(`vl_moeda_empresa` AS STRING), ''), COALESCE(CAST(`vl_moeda_objeto` AS STRING), ''), COALESCE(CAST(`tp_transacao_comercial` AS STRING), ''), COALESCE(CAST(`cod_area_contabilidade_custos` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_item_partida_individual_ledger` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`vl_moeda_global` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`qt_lancada` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`desc_texto_item` AS STRING), ''), COALESCE(CAST(`desc_classe_custo` AS STRING), ''), COALESCE(CAST(`tp_operacao_co` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`cod_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`desc_conta_razao` AS STRING), ''), COALESCE(CAST(`desc_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`desc_contrapartida` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`cod_ledger_fonte` AS STRING), ''), COALESCE(CAST(`cod_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_moeda_empresa` AS STRING), ''), COALESCE(CAST(`cod_moeda_objeto_co` AS STRING), ''), COALESCE(CAST(`cod_moeda_relatorio` AS STRING), ''), COALESCE(CAST(`nm_usuario_criou_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_objeto_relatorio` AS STRING), ''), COALESCE(CAST(`cod_periodo_contabil` AS STRING), ''), COALESCE(CAST(`tp_conta` AS STRING), ''), COALESCE(CAST(`tp_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`tp_objeto_controlling` AS STRING), ''), COALESCE(CAST(`vl_moeda_transacao` AS STRING), ''), COALESCE(CAST(`vl_moeda_empresa` AS STRING), ''), COALESCE(CAST(`vl_moeda_objeto` AS STRING), ''), COALESCE(CAST(`tp_transacao_comercial` AS STRING), ''), COALESCE(CAST(`cod_area_contabilidade_custos` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1;

## 15. Amostra de linhas completas

In [0]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
ORDER BY `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `cod_item_partida_individual_ledger`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cod_centro_custo
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro_custo` AS STRING)), ''), '(vazio)') AS `cod_centro_custo`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro_custo` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)') AS `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_periodo_contabil
SELECT COALESCE(NULLIF(trim(CAST(`cod_periodo_contabil` AS STRING)), ''), '(vazio)') AS `cod_periodo_contabil`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_periodo_contabil` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_objeto_controlling
SELECT COALESCE(NULLIF(trim(CAST(`tp_objeto_controlling` AS STRING)), ''), '(vazio)') AS `tp_objeto_controlling`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_objeto_controlling` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_conta_razao
SELECT COALESCE(NULLIF(trim(CAST(`cod_conta_razao` AS STRING)), ''), '(vazio)') AS `cod_conta_razao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_conta_razao` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [0]:
-- 17. FRESHNESS
SELECT `dt_ingestao` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY `dt_ingestao`
ORDER BY data_carga DESC
LIMIT 30;

In [0]:
-- 17.1 SNAPSHOT OU HISTORICO?
SELECT MIN(`dt_ingestao`)            AS primeira_carga,
       MAX(`dt_ingestao`)            AS ultima_carga,
       COUNT(DISTINCT `dt_ingestao`) AS cargas_distintas,
       CASE WHEN COUNT(DISTINCT `dt_ingestao`) = 1
            THEN 'SNAPSHOT - substitui a cada carga; chave NAO precisa da data'
            ELSE 'HISTORICO - acumula; a chave DEVE incluir a data de carga' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1;

## 18. Análises específicas — KSB1

### 18.1 Colunas redundantes declaradas no schema

O schema afirma que:

- `desc_conta_razao` resolve para o **mesmo texto** de `desc_classe_custo`
- `desc_contrapartida` resolve para o **mesmo texto** de `desc_conta_contrapartida`

Se confirmado, são 2 colunas redundantes que não precisariam existir —
e que no extrato do SAP aparecem como colunas distintas do relatório.

In [0]:
-- 18.1 COLUNAS REDUNDANTES
SELECT 'desc_conta_razao vs desc_classe_custo' AS par,
       COUNT(*)                                                       AS linhas,
       COUNT_IF(COALESCE(desc_conta_razao,'') = COALESCE(desc_classe_custo,''))  AS identicos,
       COUNT_IF(COALESCE(desc_conta_razao,'') <> COALESCE(desc_classe_custo,'')) AS divergentes,
       CASE WHEN COUNT_IF(COALESCE(desc_conta_razao,'') <> COALESCE(desc_classe_custo,'')) = 0
            THEN 'REDUNDANCIA CONFIRMADA' ELSE 'colunas divergem' END AS veredito
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
UNION ALL
SELECT 'desc_contrapartida vs desc_conta_contrapartida',
       COUNT(*),
       COUNT_IF(COALESCE(desc_contrapartida,'') = COALESCE(desc_conta_contrapartida,'')),
       COUNT_IF(COALESCE(desc_contrapartida,'') <> COALESCE(desc_conta_contrapartida,'')),
       CASE WHEN COUNT_IF(COALESCE(desc_contrapartida,'') <> COALESCE(desc_conta_contrapartida,'')) = 0
            THEN 'REDUNDANCIA CONFIRMADA' ELSE 'colunas divergem' END
  FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1;

### 18.2 `cod_objeto_relatorio` — preenchido com centro de custo

O schema declara que esta coluna é preenchida com o **centro de custo**, não com o objeto
de controlling. Esta célula confirma se os valores são idênticos a `cod_centro_custo`.

In [0]:
-- 18.2 cod_objeto_relatorio vs cod_centro_custo
SELECT COUNT(*)                                                             AS linhas,
       COUNT_IF(trim(COALESCE(cod_objeto_relatorio,''))
              = trim(COALESCE(cod_centro_custo,'')))                        AS identicos,
       COUNT_IF(trim(COALESCE(cod_objeto_relatorio,''))
             <> trim(COALESCE(cod_centro_custo,'')))                        AS divergentes,
       COUNT_IF(cod_objeto_relatorio IS NULL OR trim(cod_objeto_relatorio) = '') AS objeto_vazio,
       CASE WHEN COUNT_IF(trim(COALESCE(cod_objeto_relatorio,''))
                       <> trim(COALESCE(cod_centro_custo,''))) = 0
            THEN 'CONFIRMADO - identico ao centro de custo'
            ELSE 'DIVERGE - ha valores diferentes' END                      AS veredito
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1;

### 18.3 `tp_objeto_controlling` — origem não localizada

O schema declara que este campo é derivado dos 2 primeiros caracteres de
`cod_objeto_controlling` — **mas essa coluna não existe na tabela**.

O domínio declarado é: `KS`=CCS, `OR`=ORD, `PR`=PEP, `IM`=OL, `KL`=TAT, `NV`=TDR, `OV`=OPE.
Esta célula mostra o domínio real encontrado.

In [0]:
-- 18.3 DOMINIO DE tp_objeto_controlling
SELECT COALESCE(NULLIF(trim(tp_objeto_controlling), ''), '(vazio)') AS tp_objeto_controlling,
       COUNT(*)                                  AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct,
       CASE WHEN trim(tp_objeto_controlling) IN ('CCS','ORD','PEP','OL','TAT','TDR','OPE')
            THEN 'dominio declarado' ELSE 'FORA DO DOMINIO DECLARADO' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY COALESCE(NULLIF(trim(tp_objeto_controlling), ''), '(vazio)'),
         trim(tp_objeto_controlling)
ORDER BY linhas DESC;

### 18.4 Coerência entre as quatro colunas de valor

A tabela tem 4 valores monetários em moedas diferentes mais a quantidade.
Esta célula verifica o comportamento conjunto.

In [0]:
-- 18.4 COERENCIA DOS VALORES
SELECT COUNT(*)                                                  AS linhas,
       COUNT_IF(vl_moeda_transacao = vl_moeda_empresa)           AS transacao_igual_empresa,
       COUNT_IF(vl_moeda_empresa = vl_moeda_objeto)              AS empresa_igual_objeto,
       COUNT_IF(vl_moeda_global = vl_moeda_empresa)              AS global_igual_empresa,
       COUNT_IF(vl_moeda_transacao = 0)                          AS transacao_zerada,
       COUNT_IF(qt_lancada = 0)                                  AS quantidade_zerada,
       COUNT_IF(cod_debito_credito = 'S' AND vl_moeda_empresa < 0) AS debito_com_valor_negativo,
       COUNT_IF(cod_debito_credito = 'H' AND vl_moeda_empresa > 0) AS credito_com_valor_positivo,
       ROUND(SUM(vl_moeda_empresa), 2)                           AS soma_moeda_empresa
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1;

### 18.5 Distribuição por centro de custo

Esta tabela **não tem centro/planta** — o recorte natural é por centro de custo.

In [0]:
-- 18.5 CENTROS DE CUSTO
SELECT cod_centro_custo,
       COUNT(*)                             AS linhas,
       COUNT(DISTINCT num_lancamento_contabil) AS lancamentos,
       ROUND(SUM(vl_moeda_empresa), 2)      AS valor_total,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
GROUP BY cod_centro_custo
ORDER BY linhas DESC
LIMIT 50;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1
ORDER BY `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `cod_item_partida_individual_ledger`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1),
g AS (
  SELECT 'num_lancamento_contabil' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
UNION ALL
  SELECT 'num_lancamento_contabil + cod_item_partida_individual_ledger' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_lancamento_contabil`, `cod_item_partida_individual_ledger` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil + cod_item_partida_individual_ledger' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `cod_item_partida_individual_ledger` FROM dev_procurement.corp_curated.vw_ds_ctrl_ksb1)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'KSB1' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'vw_ds_ctrl_ksb1'
UNION ALL SELECT 'IDENTIFICACAO', 'tipo de objeto', 'VIEW'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '40'
UNION ALL SELECT 'IDENTIFICACAO', 'coluna de recorte', 'cod_centro_custo'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Gerar o notebook de cenário com esse recorte.
3. Extrair a KSB1 no SAP com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Formato de data normalizado (seção 12)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] Alertas de partida verificados (seção 18)
